In [28]:
from datasets import load_dataset
from src.language_models.model import RNNModel as lstm
import torch
from src.language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F




# Load dataset

In [3]:

ds = load_dataset("nyu-mll/blimp", "adjunct_island")

Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 231755.11 examples/s]


In [17]:
ds['train'][0]

{'sentence_good': 'Who should Derek hug after shocking Richard?',
 'sentence_bad': 'Who should Derek hug Richard after shocking?',
 'field': 'syntax',
 'linguistics_term': 'island_effects',
 'UID': 'adjunct_island',
 'simple_LM_method': True,
 'one_prefix_method': False,
 'two_prefix_method': False,
 'lexically_identical': True,
 'pair_id': 0}

# Load model

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt"  # Replace with your checkpoint path
model = lstm('LSTM', 50001, 200, 650, 2, 0.2, False)
with open(checkpoint_path, 'rb') as f:
    state_dict = torch.load(f, map_location='cuda' if device =='cuda' else 'cpu')
    model.load_state_dict(state_dict)
model.to(device)
model.eval() 

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 200)
  (rnn): LSTM(200, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

# Load vocabulary and tokenize dataset

In [10]:
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)

In [18]:
class BLiMPDataset(Dataset):
    def __init__(self, blimp_subset, dictionary):
        self.dataset = load_dataset("nyu-mll/blimp", blimp_subset, split = 'train')
        self.dictionary = dictionary
        self.encoded_pairs = []

        for example in self.dataset:
            sentence_good = example['sentence_good']
            sentence_bad = example['sentence_bad']

            encoded_good = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence_good.split()]
            encoded_bad = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence_bad.split()]

            self.encoded_pairs.append({
                "sentence_good": sentence_good,
                "sentence_bad": sentence_bad,
                "encoded_good": torch.tensor(encoded_good, dtype=torch.long),
                "encoded_bad": torch.tensor(encoded_bad, dtype=torch.long),
            })

    def __len__(self):
        return len(self.encoded_pairs)

    def __getitem__(self, idx):
        return self.encoded_pairs[idx]

In [19]:
blimp = BLiMPDataset("adjunct_island", dictionary)

In [21]:
blimp.__dict__

{'dataset': Dataset({
     features: ['sentence_good', 'sentence_bad', 'field', 'linguistics_term', 'UID', 'simple_LM_method', 'one_prefix_method', 'two_prefix_method', 'lexically_identical', 'pair_id'],
     num_rows: 1000
 }),
 'dictionary': <src.language_models.dictionary_corpus.Dictionary at 0x7f5974d55a90>,
 'encoded_pairs': [{'sentence_good': 'Who should Derek hug after shocking Richard?',
   'sentence_bad': 'Who should Derek hug Richard after shocking?',
   'encoded_good': tensor([ 6995,   324, 17036, 37897,   674, 16368,    62]),
   'encoded_bad': tensor([ 6995,   324, 17036, 37897,  2306,   674,    62])},
  {'sentence_good': 'What had Theresa walked through while talking about that high school?',
   'sentence_bad': 'What had Theresa walked through that high school while talking about?',
   'encoded_good': tensor([ 3340,    68, 14451, 20181,   423,   319, 10665,   839,    34,   270,
              62]),
   'encoded_bad': tensor([ 3340,    68, 14451, 20181,   423,    34,   270,  

In [25]:
def collate_fn(batch):
    """Custom collate function to properly handle sentences as lists of strings."""
    sentence_good = [item['sentence_good'] for item in batch]  # Keep lists of words as they are
    sentence_bad = [item['sentence_bad'] for item in batch]  # Keep lists of words as they are
    encoded_good = torch.stack([item['encoded_good'] for item in batch])  # Stack tensors
    encoded_bad = torch.stack([item['encoded_bad'] for item in batch])
    
    
    return {
        "sentence good": sentence_good,  
        "sentence bad": sentence_bad,
        "encoded_good": encoded_good,
        "encoded_bad": encoded_bad,
        
    }

In [35]:
def collate_fn(batch):
    encoded_good_sequences = [item['encoded_good'] for item in batch]
    encoded_bad_sequences = [item['encoded_bad'] for item in batch]
    return {
        'encoded_good': pad_sequence(encoded_good_sequences, batch_first=True),
        'encoded_bad': pad_sequence(encoded_bad_sequences, batch_first=True)
    }


In [36]:
test_dataloader = DataLoader(blimp, batch_size=512, collate_fn = collate_fn)

In [37]:
def compute_seq_nll(data, hidden, batch_size, max_len):
    #padding
    padded_data = pad_sequence(data, batch_first=True).to(device)
    #mask
    mask = (padded_data!=0).float()
    #forward pass
    output, hidden = model(padded_data, hidden)
    #target
    targets = padded_data[:, 1:]
    #log probs
    log_probs = F.log_softmax(output[:, :-1], dim=-1)
    #nll loss
    nll_loss = F.nll_loss(
            log_probs.reshape(-1, log_probs.size(-1)),
            targets.reshape(-1),
            reduction='none'
        ).reshape(batch_size, max_len - 1)
    #mask loss
    masked_nll_loss = nll_loss * mask[:, 1:]
    # Sum the negative log-likelihood over the sequence for each example
    sequence_nll = masked_nll_loss.sum(dim=1)
    
    return -sequence_nll
    

In [38]:


model.eval()
correct_predictions = 0
total_predictions = 0
#Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in test_dataloader:
        
        good = batch['encoded_good']
        bad = batch['encoded_bad']
    
        batch_size = good.size(0)
        seq_len_good = [len(seq)for seq in good ]
        seq_len_bad = [len(seq) for seq in bad]
        max_len_both = [max(seq_len_good), max(seq_len_bad)]
        max_len = max(max_len_both)
        
        hidden_good = model.init_hidden(batch_size)
        hidden_bad = model.init_hidden(batch_size)        
        seq_nll_good = compute_seq_nll(good, hidden_good, batch_size, max_len)
        seq_nll_bad = compute_seq_nll(bad, hidden_bad, batch_size, max_len)
        predictions = (seq_nll_good < seq_nll_bad).cpu().numpy()
        correct_predictions += np.sum(predictions)
        total_predictions += batch_size

accuracy = correct_predictions / total_predictions
print(f"Accuracy on {test_dataloader.dataset.dataset.builder_name}/{test_dataloader.dataset.dataset.config_name}: {accuracy * 100:.2f}%")
        
        
        

RuntimeError: Expected hidden[0] size (2, 12, 650), got [2, 512, 650]